In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from rdkit.Chem import (
    MolFromInchi,
    MolFromSmiles,
    MolToInchi,
    MolToInchiKey,
    MolToSmiles,
    PandasTools,
)
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
# Some general utility functions


def mol_to_non_stereo_inchi(mol):
    smiles = MolToSmiles(mol, isomericSmiles=False)
    non_stereo_mol = MolFromSmiles(smiles)
    inchi = MolToInchi(non_stereo_mol)
    return inchi


def smiles_to_non_stereo_inchi(smiles):
    mol = MolFromSmiles(smiles)
    return mol_to_non_stereo_inchi(mol)

Stuff to think about:

- The comparison is done by comparing InChIs without stereochemistry. I don't think GLORYx removes stereochemistry by default. Neither do we for that matter. We therefore need to do it here.
- Any metabolite with fewer than three heavy atoms is removed. This is handled by our re-implementation, but maybe not by the original GLORYx. They might have done that as a post-processing step. Here we apply this step manually to be sure.
- Any metabolite with identical InChI to the parent compound is "ignored". Does that mean it does not count in the statistics at all?
- They write "duplicate metabolite predictions are combined by retaining the highest priority score": does that mean for each parent separately or across all parents together? Probably the former.
- They did something weird with the outcome of CYP reactions (see "Special Consideration for CYP Reactions"). I don't know exactly how that translates into code for the evaluation pipeline, so I'll figure that one out later.

# Import ground truth

In [ ]:
test_parents = PandasTools.LoadSDF(
    "../datasets/GLORYx_data/test/gloryx_test_dataset.sdf"
)
test_metabolites_GT = PandasTools.LoadSDF(
    "../datasets/GLORYx_data/test/gloryx_test_dataset_metabolites_exploded.sdf"
)

In [ ]:
len(test_parents)

In [ ]:
test_metabolites_GT["num_heavy_atoms"] = test_metabolites_GT["ROMol"].apply(
    lambda x: x.GetNumHeavyAtoms()
)
test_metabolites_GT[test_metabolites_GT["num_heavy_atoms"] < 3]

In [ ]:
test_metabolites_GT["inchi"] = test_metabolites_GT["ROMol"].apply(
    lambda x: mol_to_non_stereo_inchi(x)
)

In [ ]:
len(test_metabolites_GT), len(test_metabolites_GT.inchi.unique())

### Import results from NERDD (GLORYx)

In [ ]:
test_metabolites_NERDD_P1_P2 = PandasTools.LoadSDF(
    "../datasets/GLORYx_data/test/results_nerdd/phase_1_2/results.sdf"
)
test_metabolites_NERDD_P1 = PandasTools.LoadSDF(
    "../datasets/GLORYx_data/test/results_nerdd/phase_1/results.sdf"
)
test_metabolites_NERDD_P2 = PandasTools.LoadSDF(
    "../datasets/GLORYx_data/test/results_nerdd/phase_2/results.sdf"
)

In [ ]:
# There seems to be one output for which the SMILES is "None". That looks like an error, so we remove it.
test_metabolites_NERDD_P2 = test_metabolites_NERDD_P2[
    test_metabolites_NERDD_P2["metabolite_smiles"] != "None"
]

In [ ]:
# Again, check for any metabolites with less than 3 heavy atoms
test_metabolites_NERDD_P1_P2["num_heavy_atoms"] = test_metabolites_NERDD_P1_P2[
    "metabolite_smiles"
].apply(lambda x: MolFromSmiles(x).GetNumHeavyAtoms())
test_metabolites_NERDD_P1["num_heavy_atoms"] = test_metabolites_NERDD_P1[
    "metabolite_smiles"
].apply(lambda x: MolFromSmiles(x).GetNumHeavyAtoms())
test_metabolites_NERDD_P2["num_heavy_atoms"] = test_metabolites_NERDD_P2[
    "metabolite_smiles"
].apply(lambda x: MolFromSmiles(x).GetNumHeavyAtoms())


(
    len(
        test_metabolites_NERDD_P1_P2[
            test_metabolites_NERDD_P1_P2["num_heavy_atoms"] < 3
        ]
    ),
    len(test_metabolites_NERDD_P1[test_metabolites_NERDD_P1["num_heavy_atoms"] < 3]),
    len(test_metabolites_NERDD_P2[test_metabolites_NERDD_P2["num_heavy_atoms"] < 3]),
)

In [ ]:
# Convert SMILES to non-stereo InChI
test_metabolites_NERDD_P1_P2["inchi"] = test_metabolites_NERDD_P1_P2[
    "metabolite_smiles"
].apply(lambda x: smiles_to_non_stereo_inchi(x))
test_metabolites_NERDD_P1["inchi"] = test_metabolites_NERDD_P1[
    "metabolite_smiles"
].apply(lambda x: smiles_to_non_stereo_inchi(x))
test_metabolites_NERDD_P2["inchi"] = test_metabolites_NERDD_P2[
    "metabolite_smiles"
].apply(lambda x: smiles_to_non_stereo_inchi(x))

In [ ]:
len(test_metabolites_NERDD_P1_P2), len(test_metabolites_NERDD_P1_P2.inchi.unique())

In [ ]:
len(test_metabolites_NERDD_P1), len(test_metabolites_NERDD_P1.inchi.unique())

In [ ]:
len(test_metabolites_NERDD_P2), len(test_metabolites_NERDD_P2.inchi.unique())

There seems to be 18 duplicate metabolites coming from the Phase 1 models. We'll try and remove them if they have the same parents. In that case, we keep the version with the highest score.

In [ ]:
len([group for group in test_metabolites_NERDD_P1_P2.groupby(["inchi", "mol_id"])])

There appears to be no duplicate metabolites associated with a single parent molecule. Upon inspection, the metabolites are either small, commonplace metabolites, or metabolites originating from structurally related parents.

# Compute metrics NERDD vs. GT

In [ ]:
def compute_metrics(
    test_metabolites_GT_inchis,
    test_metabolites_inchis,
    test_metabolites_scores,
    test_metabolites_ranks,
):
    # Total number of predictions
    total_predictions = len(test_metabolites_inchis)
    # Compute true positives, false positives, and false negatives
    true_positives = len(
        set(test_metabolites_inchis).intersection(set(test_metabolites_GT_inchis))
    )
    false_positives = len(
        set(test_metabolites_inchis).difference(set(test_metabolites_GT_inchis))
    )
    false_negatives = len(
        set(test_metabolites_GT_inchis).difference(set(test_metabolites_inchis))
    )
    # Compute recall and precision
    recall = (
        true_positives / (true_positives + false_negatives)
        if (true_positives + false_negatives) > 0
        else 0
    )
    precision = (
        true_positives / (true_positives + false_positives)
        if (true_positives + false_positives) > 0
        else 0
    )
    # Compute the AUC (score-based and rank-based)
    if total_predictions > 0:
        GT_binary = [
            1 if inchi in test_metabolites_GT_inchis else 0
            for inchi in test_metabolites_inchis
        ]
        auc_score_based = roc_auc_score(GT_binary, test_metabolites_scores)
        roc_curve_data = roc_curve(GT_binary, test_metabolites_scores)
        # Invert ranks to get a score-like metric for AUC calculation (higher rank = better prediction)
        inverted_ranks = [-rank for rank in test_metabolites_ranks]
        auc_rank_based = roc_auc_score(GT_binary, inverted_ranks)
    else:
        auc_score_based = 0
        auc_rank_based = 0
    return (
        recall,
        precision,
        total_predictions,
        true_positives,
        auc_score_based,
        auc_rank_based,
        roc_curve_data,
    )

In [ ]:
(
    recall_P1_P2_NERDD,
    precision_P1_P2_NERDD,
    total_predictions_P1_P2_NERDD,
    true_positives_P1_P2_NERDD,
    auc_score_based_P1_P2_NERDD,
    auc_rank_based_P1_P2_NERDD,
    roc_curve_P1_P2_NERDD,
) = compute_metrics(
    test_metabolites_GT["inchi"].values,
    test_metabolites_NERDD_P1_P2["inchi"].values,
    test_metabolites_NERDD_P1_P2["priority_score"].astype(float).values,
    test_metabolites_NERDD_P1_P2["rank"].astype(int).values,
)
(
    recall_P1_NERDD,
    precision_P1_NERDD,
    total_predictions_P1_NERDD,
    true_positives_P1_NERDD,
    auc_score_based_P1_NERDD,
    auc_rank_based_P1_NERDD,
    roc_curve_P1_NERDD,
) = compute_metrics(
    test_metabolites_GT["inchi"].values,
    test_metabolites_NERDD_P1["inchi"].values,
    test_metabolites_NERDD_P1["priority_score"].astype(float).values,
    test_metabolites_NERDD_P1["rank"].astype(int).values,
)
(
    recall_P2_NERDD,
    precision_P2_NERDD,
    total_predictions_P2_NERDD,
    true_positives_P2_NERDD,
    auc_score_based_P2_NERDD,
    auc_rank_based_P2_NERDD,
    roc_curve_P2_NERDD,
) = compute_metrics(
    test_metabolites_GT["inchi"].values,
    test_metabolites_NERDD_P2["inchi"].values,
    test_metabolites_NERDD_P2["priority_score"].astype(float).values,
    test_metabolites_NERDD_P2["rank"].astype(int).values,
)

In [ ]:
print(
    "Phase 1 + Phase 2:\n\tRecall = {:.2f}, \n\tPrecision = {:.2f}, \n\tTotal Predictions = {}, \n\tTrue Positives = {}, \n\tAUC (score-based) = {:.2f}, \n\tAUC (rank-based) = {:.2f}".format(
        recall_P1_P2_NERDD,
        precision_P1_P2_NERDD,
        total_predictions_P1_P2_NERDD,
        true_positives_P1_P2_NERDD,
        auc_score_based_P1_P2_NERDD,
        auc_rank_based_P1_P2_NERDD,
    )
)
print(
    "Phase 1:\n\tRecall = {:.2f}, \n\tPrecision = {:.2f}, \n\tTotal Predictions = {}, \n\tTrue Positives = {}, \n\tAUC (score-based) = {:.2f}, \n\tAUC (rank-based) = {:.2f}".format(
        recall_P1_NERDD,
        precision_P1_NERDD,
        total_predictions_P1_NERDD,
        true_positives_P1_NERDD,
        auc_score_based_P1_NERDD,
        auc_rank_based_P1_NERDD,
    )
)
print(
    "Phase 2:\n\tRecall = {:.2f}, \n\tPrecision = {:.2f}, \n\tTotal Predictions = {}, \n\tTrue Positives = {}, \n\tAUC (score-based) = {:.2f}, \n\tAUC (rank-based) = {:.2f}".format(
        recall_P2_NERDD,
        precision_P2_NERDD,
        total_predictions_P2_NERDD,
        true_positives_P2_NERDD,
        auc_score_based_P2_NERDD,
        auc_rank_based_P2_NERDD,
    )
)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.plot(
    roc_curve_P1_P2_NERDD[0],
    roc_curve_P1_P2_NERDD[1],
    label="Phase 1 + Phase 2 (AUC = {:.2f})".format(auc_score_based_P1_P2_NERDD),
)
plt.plot(
    roc_curve_P1_NERDD[0],
    roc_curve_P1_NERDD[1],
    label="Phase 1 (AUC = {:.2f})".format(auc_score_based_P1_NERDD),
)
plt.plot(
    roc_curve_P2_NERDD[0],
    roc_curve_P2_NERDD[1],
    label="Phase 2 (AUC = {:.2f})".format(auc_score_based_P2_NERDD),
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (based on the scores) for NERDD Predictions")
plt.legend(loc="lower right")
plt.show()

# Import results from GLORYxR

In [ ]:
test_metabolites_GLORYxR_P1_P2 = pd.read_csv(
    "../datasets/GLORYx_data/test/results_loose_priority/phase_1_2/results.csv"
)
test_metabolites_GLORYxR_P1 = pd.read_csv(
    "../datasets/GLORYx_data/test/results_loose_priority/phase_1/results.csv"
)
test_metabolites_GLORYxR_P2 = pd.read_csv(
    "../datasets/GLORYx_data/test/results_loose_priority/phase_2/results.csv"
)

In [ ]:
test_metabolites_GLORYxR_P1_P2["inchi"] = test_metabolites_GLORYxR_P1_P2[
    "metabolite_smiles"
].apply(lambda x: smiles_to_non_stereo_inchi(x))
test_metabolites_GLORYxR_P1["inchi"] = test_metabolites_GLORYxR_P1[
    "metabolite_smiles"
].apply(lambda x: smiles_to_non_stereo_inchi(x))
test_metabolites_GLORYxR_P2["inchi"] = test_metabolites_GLORYxR_P2[
    "metabolite_smiles"
].apply(lambda x: smiles_to_non_stereo_inchi(x))

In [ ]:
len(test_metabolites_GLORYxR_P1_P2), len(test_metabolites_GLORYxR_P1_P2.inchi.unique())

In [ ]:
len(test_metabolites_GLORYxR_P1), len(test_metabolites_GLORYxR_P1.inchi.unique())

In [ ]:
len(test_metabolites_GLORYxR_P2), len(test_metabolites_GLORYxR_P2.inchi.unique())

In [ ]:
test_metabolites_GLORYxR_P1_P2["parent_key"] = test_metabolites_GLORYxR_P1_P2[
    "parent_smiles"
].apply(lambda x: MolToInchiKey(MolFromSmiles(x)))
test_metabolites_GLORYxR_P1["parent_key"] = test_metabolites_GLORYxR_P1[
    "parent_smiles"
].apply(lambda x: MolToInchiKey(MolFromSmiles(x)))
test_metabolites_GLORYxR_P2["parent_key"] = test_metabolites_GLORYxR_P2[
    "parent_smiles"
].apply(lambda x: MolToInchiKey(MolFromSmiles(x)))

print(len(test_metabolites_GLORYxR_P1_P2.groupby(["parent_key"]).size()))

test_metabolites_GLORYxR_P1_P2["rank"] = test_metabolites_GLORYxR_P1_P2.groupby(
    "parent_key"
)["score"].rank(method="dense", ascending=False)
test_metabolites_GLORYxR_P1["rank"] = test_metabolites_GLORYxR_P1.groupby("parent_key")[
    "score"
].rank(method="dense", ascending=False)
test_metabolites_GLORYxR_P2["rank"] = test_metabolites_GLORYxR_P2.groupby("parent_key")[
    "score"
].rank(method="dense", ascending=False)

# Compute metrics GLORYxR vs. GT

In [ ]:
(
    recall_P1_P2_GLORYxR,
    precision_P1_P2_GLORYxR,
    total_predictions_P1_P2_GLORYxR,
    true_positives_P1_P2_GLORYxR,
    auc_score_based_P1_P2_GLORYxR,
    auc_rank_based_P1_P2_GLORYxR,
    roc_curve_P1_P2_GLORYxR,
) = compute_metrics(
    test_metabolites_GT["inchi"].values,
    test_metabolites_GLORYxR_P1_P2["inchi"].values,
    test_metabolites_GLORYxR_P1_P2["score"].astype(float).values,
    test_metabolites_GLORYxR_P1_P2["rank"].values,
)
(
    recall_P1_GLORYxR,
    precision_P1_GLORYxR,
    total_predictions_P1_GLORYxR,
    true_positives_P1_GLORYxR,
    auc_score_based_P1_GLORYxR,
    auc_rank_based_P1_GLORYxR,
    roc_curve_P1_GLORYxR,
) = compute_metrics(
    test_metabolites_GT["inchi"].values,
    test_metabolites_GLORYxR_P1["inchi"].values,
    test_metabolites_GLORYxR_P1["score"].astype(float).values,
    test_metabolites_GLORYxR_P1["rank"].values,
)
(
    recall_P2_GLORYxR,
    precision_P2_GLORYxR,
    total_predictions_P2_GLORYxR,
    true_positives_P2_GLORYxR,
    auc_score_based_P2_GLORYxR,
    auc_rank_based_P2_GLORYxR,
    roc_curve_P2_GLORYxR,
) = compute_metrics(
    test_metabolites_GT["inchi"].values,
    test_metabolites_GLORYxR_P2["inchi"].values,
    test_metabolites_GLORYxR_P2["score"].astype(float).values,
    test_metabolites_GLORYxR_P2["rank"].values,
)

In [ ]:
print(
    "Phase 1 + Phase 2:\n\tRecall = {:.2f}, \n\tPrecision = {:.2f}, \n\tTotal Predictions = {}, \n\tTrue Positives = {}, \n\tAUC (score-based) = {:.2f}, \n\tAUC (rank-based) = {:.2f}".format(
        recall_P1_P2_GLORYxR,
        precision_P1_P2_GLORYxR,
        total_predictions_P1_P2_GLORYxR,
        true_positives_P1_P2_GLORYxR,
        auc_score_based_P1_P2_GLORYxR,
        auc_rank_based_P1_P2_GLORYxR,
    )
)
print(
    "Phase 1:\n\tRecall = {:.2f}, \n\tPrecision = {:.2f}, \n\tTotal Predictions = {}, \n\tTrue Positives = {}, \n\tAUC (score-based) = {:.2f}, \n\tAUC (rank-based) = {:.2f}".format(
        recall_P1_GLORYxR,
        precision_P1_GLORYxR,
        total_predictions_P1_GLORYxR,
        true_positives_P1_GLORYxR,
        auc_score_based_P1_GLORYxR,
        auc_rank_based_P1_GLORYxR,
    )
)
print(
    "Phase 2:\n\tRecall = {:.2f}, \n\tPrecision = {:.2f}, \n\tTotal Predictions = {}, \n\tTrue Positives = {}, \n\tAUC (score-based) = {:.2f}, \n\tAUC (rank-based) = {:.2f}".format(
        recall_P2_GLORYxR,
        precision_P2_GLORYxR,
        total_predictions_P2_GLORYxR,
        true_positives_P2_GLORYxR,
        auc_score_based_P2_GLORYxR,
        auc_rank_based_P2_GLORYxR,
    )
)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.plot(
    roc_curve_P1_P2_GLORYxR[0],
    roc_curve_P1_P2_GLORYxR[1],
    label="Phase 1 + Phase 2 (AUC = {:.2f})".format(auc_score_based_P1_P2_GLORYxR),
)
plt.plot(
    roc_curve_P1_GLORYxR[0],
    roc_curve_P1_GLORYxR[1],
    label="Phase 1 (AUC = {:.2f})".format(auc_score_based_P1_GLORYxR),
)
plt.plot(
    roc_curve_P2_GLORYxR[0],
    roc_curve_P2_GLORYxR[1],
    label="Phase 2 (AUC = {:.2f})".format(auc_score_based_P2_GLORYxR),
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (based on the scores) for GLORYxR Predictions")
plt.legend(loc="lower right")
plt.show()

### Compare NERDD AND GLORYxR outputs directly

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].plot(
    roc_curve_P1_P2_NERDD[0],
    roc_curve_P1_P2_NERDD[1],
    label="NERDD Phase 1 + Phase 2 (AUC = {:.2f})".format(auc_score_based_P1_P2_NERDD),
)
axs[0].plot(
    roc_curve_P1_P2_GLORYxR[0],
    roc_curve_P1_P2_GLORYxR[1],
    label="GLORYxR Phase 1 + Phase 2 (AUC = {:.2f})".format(
        auc_score_based_P1_P2_GLORYxR
    ),
)
axs[0].set_xlabel("False Positive Rate")
axs[0].set_ylabel("True Positive Rate")
axs[0].set_title("Phase 1 + Phase 2")
axs[0].legend(loc="lower right")
axs[1].plot(
    roc_curve_P1_NERDD[0],
    roc_curve_P1_NERDD[1],
    label="NERDD Phase 1 (AUC = {:.2f})".format(auc_score_based_P1_NERDD),
)
axs[1].plot(
    roc_curve_P1_GLORYxR[0],
    roc_curve_P1_GLORYxR[1],
    label="GLORYxR Phase 1 (AUC = {:.2f})".format(auc_score_based_P1_GLORYxR),
)
axs[1].set_xlabel("False Positive Rate")
axs[1].set_ylabel("True Positive Rate")
axs[1].set_title("Phase 1")
axs[1].legend(loc="lower right")
axs[2].plot(
    roc_curve_P2_NERDD[0],
    roc_curve_P2_NERDD[1],
    label="NERDD Phase 2 (AUC = {:.2f})".format(auc_score_based_P2_NERDD),
)
axs[2].plot(
    roc_curve_P2_GLORYxR[0],
    roc_curve_P2_GLORYxR[1],
    label="GLORYxR Phase 2 (AUC = {:.2f})".format(auc_score_based_P2_GLORYxR),
)
axs[2].set_xlabel("False Positive Rate")
axs[2].set_ylabel("True Positive Rate")
axs[2].set_title("Phase 2")
axs[2].legend(loc="lower right")
plt.show()

In [ ]:
metabolites_inchi_NERDD_P1_P2 = test_metabolites_NERDD_P1_P2.inchi.values
metabolites_inchi_GLORYxR_P1_P2 = test_metabolites_GLORYxR_P1_P2.inchi.values

In [ ]:
len(metabolites_inchi_NERDD_P1_P2), len(metabolites_inchi_GLORYxR_P1_P2)

We somehow obtain more metabolites than GLORYx (1795 vs. 1721) when using the "test set". Why is that? Are we missing some sort of post-processing? Btw., according to the paper, we should obtain 1724. Thus, the version on NERDD appears to be already slightly different than what was used in the paper. This could be explained by the fact that NERDD adds some internal preprocessing/postprocessing steps.

In [ ]:
len(set(metabolites_inchi_NERDD_P1_P2)), len(set(metabolites_inchi_GLORYxR_P1_P2))

In [ ]:
len(
    set(metabolites_inchi_NERDD_P1_P2).intersection(
        set(metabolites_inchi_GLORYxR_P1_P2)
    )
)

In [ ]:
inchi_NERDD_MINUS_GLORYxR = set(metabolites_inchi_NERDD_P1_P2).difference(
    set(metabolites_inchi_GLORYxR_P1_P2)
)
inchi_GLORYxR_MINUS_NERDD = set(metabolites_inchi_GLORYxR_P1_P2).difference(
    set(metabolites_inchi_NERDD_P1_P2)
)

In [ ]:
NERDD_MINUS_GLORYxR = test_metabolites_NERDD_P1_P2[
    test_metabolites_NERDD_P1_P2.inchi.isin(inchi_NERDD_MINUS_GLORYxR)
]
GLORYxR_MINUS_NERDD = test_metabolites_GLORYxR_P1_P2[
    test_metabolites_GLORYxR_P1_P2.inchi.isin(inchi_GLORYxR_MINUS_NERDD)
]

In [ ]:
len(NERDD_MINUS_GLORYxR), len(GLORYxR_MINUS_NERDD)

In [ ]:
set(NERDD_MINUS_GLORYxR.reaction_type.values)

In [ ]:
GLORYxR_MINUS_NERDD.iloc[0].parent_smiles

In [ ]:
test = MolFromSmiles("CCCNCC")  # C1CCN(C(=O)C1)C2=CC=CC=C2
display(test)
rxn = Chem.ReactionFromSmarts("[N:1][C:2]>>([N:1].[C:2]=O)")
display(rxn)
display(*(mol for mol in itertools.chain.from_iterable(rxn.RunReactants([test]))))

In [ ]:
rxn

In [ ]:
Chem.ReactionToSmarts(rxn)

In [ ]:
import itertools

from rdkit.Chem import AllChem as Chem

In [ ]:
display(
    *(
        Chem.RemoveHs(mol)
        for mol in itertools.chain.from_iterable(rxn.RunReactants([Chem.AddHs(test)]))
    )
)

In [ ]:
MolFromSmiles(GLORYxR_MINUS_NERDD.iloc[0].metabolite_smiles)

In [ ]:
GLORYxR_MINUS_NERDD.iloc[0]

In [ ]:
Chem.ReactionFromSmarts("[C:1]([H])-[C:2]([H])-[C:3]=[O:4]>>[C:1]=[C:2].[C:3]=[O:4]")

In [ ]:
rxn = [
    rxn
    for rxn in reactor.abstract_reactions
    if rxn.GetProp("_Name") == "N-dealkylation"
][0]


In [ ]:
reactor = Reactor(phase=1)

In [ ]:
from gloryxr import Reactor
from gloryxr.reactions import _to_concrete_reactions

In [ ]:
	SMIRKS
9	[NX3:2][CX3;H1]=O>>[N:2]

In [ ]:
set(GLORYxR_MINUS_NERDD.reaction_type.values)

All of the metabolites that are in the output of one model but not in the output of the other come from CYP-mediated reactions (CYP rules from GLORY (phase 1)), except the one with the rule "N-acetylation_(NH1-CH3)"; that is a NAT-mediated reaction. That might be explained by the weird thing they mentioned doing to the outcome of CYP reactions. TODO: investigate further.

General notes:

- The loose setting with the SyGMa priorities yields results that are very close to the GLORYx results. Only the AUC seems a bit lower. This might have somethign to do with the extra metabolites that we get. TODO: investigate.

- Correcting the priorities of the "phase 1 rules from SyGMa" from common to uncommon (as written in the paper), yields worse results. They probably didn't do that.

- Removing the SyGMa priorities altogether yields worse results. We'd best keep them.